In [1]:
!pip install pyvi

In [25]:

import torch

data_path = "IWSLT'15 en-vi/"
train_data_path = "/kaggle/input/iwslt15-englishvietnamese/IWSLT'15 en-vi/"
saved_model_path = '/kaggle/working/'
saved_tokenizer_path = '/kaggle/working/'
test_data_path = 'data/test_data/'

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

MAX_SEQ_LEN = 60
ENGLISH_VOCAB_SIZE = 40000  
VIETNAMESE_VOCAB_SIZE = 50000  

NUM_LAYERS = 6
D_MODEL = 512
D_FF = 2048
EPS = 0.1
BATCH_SIZE = 128
NUM_HEADS = 8
EPOCHS = 25
DROPOUT = 0.2
CLIP = 1.0
BATCH_PRINT = 100



UNKNOWN_TOKEN = '<unk>'
PAD_TOKEN = '<pad>'
START_TOKEN = '<start>'
END_TOKEN = '<end>'
PAD_TOKEN_POS = 0

# Tokenizing

500K dataset: Use 32K shared vocab with BPE/SentencePiece


1M dataset: Use 40K-50K shared vocab with BPE/SentencePiece

In [3]:
def get_tokenize(data, add_start_end=False, max_vocab_size=None):
    """Create tokenizer with vocabulary size limit"""
    tokenizer = Tokenizer(
        filters='', 
        oov_token=UNKNOWN_TOKEN
    )
    
    if add_start_end:
        tokenizer.fit_on_texts([START_TOKEN, END_TOKEN] + data)
    else:
        tokenizer.fit_on_texts(data)
    
    # MANUALLY LIMIT VOCABULARY SIZE
    if max_vocab_size is not None:
        # Sort words by frequency (most frequent first)
        sorted_words = sorted(tokenizer.word_counts.items(), key=lambda x: x[1], reverse=True)
        
        # Keep only top max_vocab_size words
        top_words = dict(sorted_words[:max_vocab_size])
        
        # Rebuild word_index with limited vocabulary
        new_word_index = {UNKNOWN_TOKEN: 1}  # oov token
        if add_start_end:
            new_word_index[START_TOKEN] = len(new_word_index) + 1
            new_word_index[END_TOKEN] = len(new_word_index) + 1
        
        idx = len(new_word_index) + 1
        for word in top_words:
            if word not in new_word_index:
                new_word_index[word] = idx
                idx += 1
        
        tokenizer.word_index = new_word_index
        tokenizer.index_word = {v: k for k, v in new_word_index.items()}
        
        print(f"   Vocabulary reduced to {len(new_word_index):,} tokens")
    
    return data, tokenizer

In [4]:
def get_tokenize_seq(en_data, vi_data, en_tokenizer, vi_tokenizer, max_sequence_length):
    en_data = [f"{START_TOKEN} {sentence} {END_TOKEN}" for sentence in en_data]
    en_sequences = en_tokenizer.texts_to_sequences(en_data)
    vi_data = [ViTokenizer.tokenize(sentence) for sentence in vi_data]
    vi_sequences = vi_tokenizer.texts_to_sequences(vi_data)
    
    filtered_en = []
    filtered_vi = []
    for i in range(len(en_sequences)):
        if (len(en_sequences[i]) <= max_sequence_length) and (len(vi_sequences[i]) <= max_sequence_length):
            filtered_en.append(en_sequences[i])
            filtered_vi.append(vi_sequences[i])
    
    return filtered_en, filtered_vi

def preprocess_tokenizer(en_data, vi_data):
    print("Creating English tokenizer (limit: 40,000)...")
    en_data, en_tokenizer = get_tokenize(
        en_data, 
        add_start_end=True,
        max_vocab_size=ENGLISH_VOCAB_SIZE  
    )
    
    print("Creating Vietnamese tokenizer (limit: 50,000)...")
    vi_data = [ViTokenizer.tokenize(sentence) for sentence in vi_data]
    vi_data, vi_tokenizer = get_tokenize(
        vi_data,
        max_vocab_size=VIETNAMESE_VOCAB_SIZE  
    )
    
    return en_tokenizer, vi_tokenizer



# Dataset

In [5]:
import torch
from torch.utils.data import Dataset, DataLoader
from pyvi.ViTokenizer import ViTokenizer
from keras.src.legacy.preprocessing.text import Tokenizer
from keras.src.utils import pad_sequences

class TranslationDataset(Dataset):
    def __init__(self, en_sequences, vi_sequences, max_length):
        self.en_sequences = en_sequences
        self.vi_sequences = vi_sequences
        self.max_length = max_length
        
    def __len__(self):
        return len(self.en_sequences)
    
    def __getitem__(self, idx):
        en_seq = self.en_sequences[idx]
        vi_seq = self.vi_sequences[idx]
        
        # Pad to max_length
        if len(en_seq) < self.max_length:
            en_seq = en_seq + [PAD_TOKEN_POS] * (self.max_length - len(en_seq))
        if len(vi_seq) < self.max_length:
            vi_seq = vi_seq + [PAD_TOKEN_POS] * (self.max_length - len(vi_seq))
            
        return torch.tensor(vi_seq, dtype=torch.long), torch.tensor(en_seq, dtype=torch.long)



In [6]:
def load_data(en_file, vi_file):
    with open(en_file,'r', encoding = 'utf-8') as f:
        en_data = f.read().strip().split("\n")
    with open(vi_file,'r', encoding = 'utf-8') as f:
        vi_data = f.read().strip().split("\n")
    return en_data, vi_data

In [7]:
def preprocess_data(train_src_path, train_trg_path, val_src_path, val_trg_path):
    en_data, vi_data = load_data(train_src_path, train_trg_path)
    en_data_val, vi_data_val = load_data(val_src_path, val_trg_path)
    
    en_tokenizer, vi_tokenizer = preprocess_tokenizer(en_data, vi_data)
    
    en_sequences, vi_sequences = get_tokenize_seq(
        en_data, vi_data, en_tokenizer, vi_tokenizer, max_sequence_length=MAX_SEQ_LEN
    )
    en_val_sequences, vi_val_sequences = get_tokenize_seq(
        en_data_val, vi_data_val, en_tokenizer, vi_tokenizer, max_sequence_length=MAX_SEQ_LEN
    )
    
    all_train_sequences = TranslationDataset(en_sequences, vi_sequences, MAX_SEQ_LEN)
    all_val_sequences = TranslationDataset(en_val_sequences, vi_val_sequences, MAX_SEQ_LEN)
    
    return en_tokenizer, vi_tokenizer, all_train_sequences, all_val_sequences

# Multi Head Attention

In [8]:
from torch import nn


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()

        assert d_model% num_heads==0
        
        self.num_heads = num_heads
        
        self.attention = ScaleDotProductAttention()
        
        self.w_q = nn.Linear(d_model, d_model)
        
        self.w_k = nn.Linear(d_model, d_model)
        
        self.w_v = nn.Linear(d_model, d_model)
        
        self.w_concat = nn.Linear(d_model, d_model)

    def forward(self, query, key, value, mask=None):
        query, key, value = self.w_q(query), self.w_k(key), self.w_v(value)

        query, key, value = self.split(query), self.split(key), self.split(value)

        out, attention = self.attention(query, key, value, mask=mask)

        out = self.concat(out)
        out = self.w_concat(out)

        return out

    def split(self, tensor):
        batch_size, length, d_model = tensor.size()

        d_tensor = d_model // self.num_heads
        tensor = tensor.view(batch_size, length, self.num_heads, d_tensor).transpose(1, 2)

        return tensor

    def concat(self, tensor):
        batch_size, num_heads, length, d_tensor = tensor.size()
        d_model = d_tensor * self.num_heads

        tensor = tensor.transpose(1, 2).contiguous().view(batch_size, length, d_model)
        return tensor

# Scaled Dot Product

In [9]:
import torch
from torch import nn
import math

class ScaleDotProductAttention(nn.Module):
    def __init__(self):
        super(ScaleDotProductAttention, self).__init__()
        self.softmax = nn.Softmax(dim=-1)
    
    def forward(self, query, key, value, mask=None):
        batch_size, num_heads, length, d_tensor = key.size()
        
        key_t = key.transpose(2, 3)
        score = (query @ key_t) / math.sqrt(d_tensor)
        
        if mask is not None:
            score = score.masked_fill(mask == 0, -1e4)
        
        score = self.softmax(score)
        
        value = score @ value
        
        return value, score


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len, device):
        """
           constructor of sinusoid encoding class

           :param d_model: dimension of model
           :param max_len: max sequence length
           :param device: hardware device setting
        """
        super(PositionalEncoding, self).__init__()

        # same size with input matrix (for adding with input matrix)
        self.encoding = torch.zeros(max_len, d_model, device=device)
        self.encoding.requires_grad = False # we don't need to compute gradient

        pos = torch.arange(0, max_len, device=device)
        pos = pos.float().unsqueeze(dim=1)

        _2i = torch.arange(0, d_model, 2, device=device).float()

        self.encoding[:, 0::2] = torch.sin(pos / (10000 ** (_2i / d_model)))
        self.encoding[:, 1::2] = torch.cos(pos / (10000 ** (_2i / d_model)))
        # compute positional encoding to consider positional information of words

    def forward(self, x):
        batch_size, seq_len = x.size()
        return self.encoding[:seq_len, :]

class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super(PositionwiseFeedForward, self).__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.linear1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.linear2(x)
        return x

class TransformerEmbedding(nn.Module):
    def __init__(self, vocab_size, d_model, max_len, dropout, device):
        super(TransformerEmbedding, self).__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model, padding_idx=PAD_TOKEN_POS)
        self.pos_emb = PositionalEncoding(d_model, max_len, device)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        tok_emb = self.tok_emb(x)
        pos_emb = self.pos_emb(x)
        return self.dropout(tok_emb + pos_emb)


# Encoder

In [10]:
from torch import nn

class EncoderLayer(nn.Module):
    def __init__(self, d_model, d_ff, num_heads, dropout):
        super(EncoderLayer, self).__init__()
        self.attention = MultiHeadAttention(d_model, num_heads)
        self.norm1 = nn.LayerNorm(d_model, eps=EPS)
        self.dropout1 = nn.Dropout(dropout)

        self.ffn = PositionwiseFeedForward(d_model, d_ff, dropout)
        self.norm2 = nn.LayerNorm(d_model, eps=EPS)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x, src_mask):
        # 1. compute self attention
        _x = x
        x = self.attention(x, x, x, src_mask)

        # 2. add and norm
        x = self.dropout1(x)
        x = self.norm1(_x + x)

        # 3. positionwise feed forward network
        _x = x
        x = self.ffn(x)

        # 4. add and norm
        x = self.dropout2(x)
        x = self.norm2(_x + x)

        return x

class Encoder(nn.Module):
    def __init__(self, inp_vocab_size, max_len, d_model, d_ff, num_heads, num_layers, dropout, device):
        super(Encoder, self).__init__()
        self.emb = TransformerEmbedding(inp_vocab_size, d_model, max_len, dropout, device=device)
        self.layers = nn.ModuleList([EncoderLayer(d_model, d_ff, num_heads, dropout) for _ in range(num_layers)])

    def forward(self, src, src_mask):
        x = self.emb(src)
        for layer in self.layers:
            x = layer(x, src_mask)

        return x

# Decoder

In [11]:
from torch import nn

class Decoder_Layer(nn.Module):
    def __init__(self, d_model, d_ff, num_heads, dropout):
        super(Decoder_Layer, self).__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.norm1 = nn.LayerNorm(d_model, eps=EPS)
        self.dropout1 = nn.Dropout(dropout)

        self.enc_dec_attn = MultiHeadAttention(d_model, num_heads)
        self.norm2 = nn.LayerNorm(d_model, eps=EPS)
        self.dropout2 = nn.Dropout(dropout)

        self.ffn = PositionwiseFeedForward(d_model, d_ff, DROPOUT)
        self.norm3 = nn.LayerNorm(d_model, eps=EPS)
        self.dropout3 = nn.Dropout(dropout)

    def forward(self, x, enc_out, trg_mask, src_mask):
        # 1. compute self attention
        _x = x
        x = self.self_attn(x, x, x, mask=trg_mask)

        # 2. add and norm
        x = self.dropout1(x)
        x = self.norm1(_x + x)

        if enc_out is not None:
            # 3. compute encoder - decoder attention
            _x = x
            x = self.enc_dec_attn(x, enc_out, enc_out, mask=src_mask)

            # 4. add and norm
            x = self.dropout2(x)
            x = self.norm2(_x + x)

        # 5. positionwise feed forward network
        _x = x
        x = self.ffn(x)

        # 6. add and norm
        x = self.dropout3(x)
        x = self.norm3(_x + x)

        return x

class Decoder(nn.Module):
    def __init__(self, trg_vocab_size, max_len, d_model, d_ff, num_heads, num_layers, dropout, device):
        super(Decoder, self).__init__()
        self.embedding = TransformerEmbedding(trg_vocab_size, d_model, max_len, dropout, device)
        self.layers = nn.ModuleList([Decoder_Layer(d_model, d_ff, num_heads, dropout) for i in range(num_layers)])
        self.linear = nn.Linear(d_model, trg_vocab_size)

    def forward(self, trg, enc_src, trg_mask, src_mask):
        trg = self.embedding(trg)

        for layer in self.layers:
            trg = layer(trg, enc_src, trg_mask, src_mask)

        # pass to LM head
        output = self.linear(trg)

        return output


# Transformer

In [12]:
import torch
from torch import nn

class Transformer(nn.Module):
    def __init__(self, src_pad_idx, trg_pad_idx, inp_vocab_size, trg_vocab_size, d_model, num_heads, max_len, d_ff, num_layers, dropout, device):
        super(Transformer, self).__init__()
        self.src_pad_idx = src_pad_idx
        self.trg_pad_idx = trg_pad_idx
        self.device = device

        self.encoder = Encoder(inp_vocab_size, max_len, d_model, d_ff, num_heads, num_layers, dropout, device)
        self.decoder = Decoder(trg_vocab_size, max_len, d_model, d_ff, num_heads, num_layers, dropout, device)

    def forward(self, src, trg):
        src_mask = self.make_src_mask(src)
        trg_mask = self.make_trg_mask(trg)
        enc_out = self.encoder(src, src_mask)
        output = self.decoder(trg, enc_out, trg_mask, src_mask)
        return output

    def make_src_mask(self, src):
        src_mask = (src != self.src_pad_idx).unsqueeze(dim=1).unsqueeze(dim=2)
        return src_mask

    def make_trg_mask(self, trg):
        trg_pad_mask = (trg != self.trg_pad_idx).unsqueeze(dim=1).unsqueeze(dim=3)
        trg_len = trg.shape[1]
        trg_look_ahead_mask = torch.tril(torch.ones(trg_len, trg_len)).bool().to(self.device)
        trg_mask = trg_pad_mask & trg_look_ahead_mask

        return trg_mask

# Learning Scheduler

In [13]:
from torch.optim.lr_scheduler import _LRScheduler

class LearningRateSchedule(_LRScheduler):
    def __init__(self, optimizer, initial_lr, decay_rates, decay_steps, lr_decay_interval, last_epoch=-1):
        """
        initial_lr: Learning rate ban đầu
        decay_rates: Danh sách hệ số decay (n phần tử)
        decay_steps: Danh sách step ứng với decay (n-1 phần tử)
        lr_decay_interval: Khoảng cách giữa các lần decay
        """
        assert len(decay_rates) - 1 == len(decay_steps), "Số lượng decay_steps phải ít hơn decay_rates một phần tử"

        self.initial_lr = initial_lr
        self.decay_rates = decay_rates
        self.decay_steps = decay_steps
        self.lr_decay_interval = lr_decay_interval
        self.prev_decay_step = 0

        super().__init__(optimizer, last_epoch)

    def get_lr(self):
        step = self.last_epoch
        lr = self.initial_lr
        prev_decay_step = 0

   
        for i in range(len(self.decay_steps)):
            decay_factor = self.decay_rates[i]
            num_intervals = max((min(step, self.decay_steps[i]) - prev_decay_step) // self.lr_decay_interval, 0)
            lr *= decay_factor ** num_intervals
            prev_decay_step = self.decay_steps[i]

   
        decay_factor = self.decay_rates[-1]
        num_intervals = max((step - prev_decay_step) // self.lr_decay_interval, 0)
        lr *= decay_factor ** num_intervals

        return [lr for _ in self.base_lrs]  

    def state_dict(self):
        return {
            "initial_lr": self.initial_lr,
            "decay_rates": self.decay_rates,
            "decay_steps": self.decay_steps,
            "lr_decay_interval": self.lr_decay_interval,
            "prev_decay_step": self.prev_decay_step
        }

    def load_state_dict(self, state_dict):
        self.initial_lr = state_dict["initial_lr"]
        self.decay_rates = state_dict["decay_rates"]
        self.decay_steps = state_dict["decay_steps"]
        self.lr_decay_interval = state_dict["lr_decay_interval"]
        self.prev_decay_step = state_dict["prev_decay_step"]

In [14]:
class TransformerLRSchedule:
    """
    Custom learning rate scheduler for Transformer models.
    Implements warmup followed by decay, with optional max_lr cap.
    """
    def __init__(self, optimizer, d_model, warmup_steps, factor=1.0, max_lr=None):
        self.optimizer = optimizer
        self.d_model = d_model
        self.warmup_steps = warmup_steps
        self.factor = factor
        self.max_lr = max_lr  # Optional maximum learning rate cap
        self.current_step = 0
        
    def step(self):
        """Update learning rate based on current step"""
        self.current_step += 1
        lr = self.get_lr()
        
        for param_group in self.optimizer.param_groups:
            param_group['lr'] = lr
    
    def get_lr(self):
        """Calculate learning rate for current step"""
        step = max(self.current_step, 1)  # Avoid division by zero
        
        # Standard Transformer learning rate schedule
        lr = self.factor * (self.d_model ** -0.5) * min(
            step ** -0.5, 
            step * (self.warmup_steps ** -1.5)
        )
        
        # Apply max_lr cap if specified
        if self.max_lr is not None:
            lr = min(lr, self.max_lr)
        
        return lr
    
    def get_last_lr(self):
        """Return current learning rate (for logging)"""
        return [self.get_lr()]


WARMUP_STEPS = 1000 
FACTOR = 1.0  # Can tune this if needed (0.5-2.0 range)


# Training

In [15]:
import torch
import math
import time
import gc
from torch import nn, optim
from torch.utils.data import Dataset, DataLoader
from pyvi.ViTokenizer import ViTokenizer
from keras.src.legacy.preprocessing.text import Tokenizer

# ==================== RESUME TRAINING CONFIGURATION ====================
RESUME_TRAINING = True  
CHECKPOINT_PATH = '/kaggle/input/trained-models/model-2.760-0.455_40k_50k_4_epoches_vi_to_en.pt'
TRAINED_EPOCHS = 4     

# ==================== CRITICAL FIX: UPDATE ScaleDotProductAttention ====================
# Replace your existing ScaleDotProductAttention class with this:



# ==================== TRAINING FUNCTIONS ====================

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def epoch_time(start_time, end_time):
    elapsed_time = end_time - start_time
    elapsed_mins = int(elapsed_time / 60)
    elapsed_secs = int(elapsed_time - (elapsed_mins * 60))
    return elapsed_mins, elapsed_secs

def print_gpu_memory():
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        reserved = torch.cuda.memory_reserved() / 1024**3
        print(f'GPU Memory: {allocated:.2f}GB allocated, {reserved:.2f}GB reserved')

def train(model, iterator, optimizer, criterion, clip, scaler, scheduler, accumulation_steps=2):
    model.train()
    epoch_loss = 0
    total_correct = 0
    total_tokens = 0
    optimizer.zero_grad()
    
    for i, (src, trg) in enumerate(iterator):
        src = src.to(model.device, non_blocking=True)
        trg = trg.to(model.device, non_blocking=True)
        
        with torch.amp.autocast('cuda'):
            output = model(src, trg[:, :-1])
            output_reshape = output.contiguous().view(-1, output.shape[-1])
            trg_reshaped = trg[:, 1:].contiguous().view(-1)
            loss = criterion(output_reshape, trg_reshaped)
            loss = loss / accumulation_steps
        
        scaler.scale(loss).backward()
        
        if (i + 1) % accumulation_steps == 0:
            scaler.unscale_(optimizer)
            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()
            
            # Clear cache periodically
            if (i + 1) % (accumulation_steps * 10) == 0:
                torch.cuda.empty_cache()
        
        with torch.no_grad():
            pred = output.argmax(dim=-1).view(-1)
            mask = (trg_reshaped != PAD_TOKEN_POS)
            correct = (pred == trg_reshaped) & mask
            total_correct += correct.sum().item()
            total_tokens += mask.sum().item()
        
        epoch_loss += loss.item() * accumulation_steps
        
        if (i + 1) % BATCH_PRINT == 0:
            lr = optimizer.param_groups[0]['lr']
            acc = total_correct / total_tokens if total_tokens > 0 else 0
            print(f'Batch: {i+1}/{len(iterator)}, Loss: {loss.item() * accumulation_steps:.4f}, '
                  f'Accuracy: {acc:.4f}, LR: {lr:.6f}')
        
        del src, trg, output, output_reshape, trg_reshaped, loss, pred, mask, correct
            
    return epoch_loss / len(iterator), total_correct / total_tokens

def evaluate(model, iterator, criterion):
    model.eval()
    epoch_loss = 0
    total_correct = 0
    total_tokens = 0
    
    with torch.no_grad():
        for i, (src, trg) in enumerate(iterator):
            src = src.to(model.device, non_blocking=True)
            trg = trg.to(model.device, non_blocking=True)
            
            with torch.amp.autocast('cuda'):
                output = model(src, trg[:, :-1])
                output_reshape = output.contiguous().view(-1, output.shape[-1])
                trg_reshaped = trg[:, 1:].contiguous().view(-1)
                loss = criterion(output_reshape, trg_reshaped)

            pred = output.argmax(dim=-1).view(-1)
            mask = (trg_reshaped != PAD_TOKEN_POS)
            correct = (pred == trg_reshaped) & mask
            total_correct += correct.sum().item()
            total_tokens += mask.sum().item()
            epoch_loss += loss.item()
            
            del src, trg, output, output_reshape, trg_reshaped, loss, pred, mask, correct
            
            if (i + 1) % 50 == 0:
                torch.cuda.empty_cache()

    return epoch_loss / len(iterator), total_correct / total_tokens

def run(total_epoch, best_loss, start_epoch=0, accumulation_steps=2):
    train_losses, test_losses = [], []
    
    for step in range(start_epoch, start_epoch + total_epoch):
        print(f'\n{"="*60}')
        print(f'Epoch: {step + 1}/{start_epoch + total_epoch}')
        print(f'{"="*60}')
        
        start_time = time.time()
        
        try:
            train_loss, train_accuracy = train(
                model, train_batches, optimizer, criterion, 
                CLIP, scaler, scheduler, accumulation_steps
            )
            
            torch.cuda.empty_cache()
            gc.collect()
            
            val_loss, val_accuracy = evaluate(model, val_batches, criterion)
            
            end_time = time.time()
            epoch_mins, epoch_secs = epoch_time(start_time, end_time)
            
            train_losses.append(train_loss)
            test_losses.append(val_loss)
            
            if val_loss < best_loss:
                best_loss = val_loss
                torch.save(model.state_dict(), 
                          f'{saved_model_path}/model-{val_loss:.3f}-{val_accuracy:.3f}.pt')
                print(f'Best val loss: {best_loss:.3f}')

            print(f'\nEpoch {step + 1} Summary:')
            print(f'  Time: {epoch_mins}m {epoch_secs}s')
            print(f'  Train Loss: {train_loss:.3f} | Train Acc: {train_accuracy:.3f} | Train PPL: {math.exp(train_loss):7.3f}')
            print(f'  Val Loss: {val_loss:.3f} | Val Acc: {val_accuracy:.3f} | Val PPL: {math.exp(val_loss):7.3f}')
            
        except RuntimeError as e:
            if "out of memory" in str(e):
                print(f'\n!!! OOM Error at epoch {step + 1} !!!')
                print('Try reducing: batch size, MAX_SEQ_LEN, or model size')
                print_gpu_memory()
                torch.cuda.empty_cache()
                gc.collect()
                raise e
            else:
                raise e
        
        torch.cuda.empty_cache()
        gc.collect()
    
    return train_losses, test_losses



# Evaluate without beam search

In [16]:
# # ==================== MAIN EXECUTION ====================

# # Clear GPU cache
# torch.cuda.empty_cache()
# gc.collect()

# print("="*60)
# print("PREPROCESSING DATA")
# print("="*60)

# # Load and preprocess data
# en_tokenizer, vi_tokenizer, all_train_sequences, all_val_sequences = preprocess_data(
#     "/kaggle/input/train-en-vi/train_2456580.en",
#     "/kaggle/input/train-en-vi/train_2456580.vi",
#     data_path + "tst2013.en.txt", 
#     data_path + "tst2013.vi.txt"
# )

# # Create DataLoaders with optimized batch size
# REDUCED_BATCH_SIZE = 128  # Increased from 8 due to vocab reduction

# # ==================== SAVE TOKENIZERS ====================
# print("\nSaving tokenizers...")
# import pickle

# try:
#     with open(f'{saved_tokenizer_path}/en_tokenizer.pkl', 'wb') as f:
#         pickle.dump(en_tokenizer, f)
        
#     with open(f'{saved_tokenizer_path}/vi_tokenizer.pkl', 'wb') as f:
#         pickle.dump(vi_tokenizer, f)
        
#     print("✓ Tokenizers saved to:", saved_tokenizer_path)
# except Exception as e:
#     print(f"⚠ Warning: Could not save tokenizers: {e}")

# # Create DataLoaders with optimized batch size

# train_batches = DataLoader(
#     all_train_sequences, 
#     batch_size=REDUCED_BATCH_SIZE, 
#     shuffle=True,
#     pin_memory=False,
#     num_workers=0
# )
# val_batches = DataLoader(
#     all_val_sequences, 
#     batch_size=REDUCED_BATCH_SIZE, 
#     shuffle=False,
#     pin_memory=False,
#     num_workers=0
# )

# # Get vocab sizes
# en_vocab_size = len(en_tokenizer.word_index) + 1
# vi_vocab_size = len(vi_tokenizer.word_index) + 1

# print(f"\nVocabulary sizes:")
# print(f"  English: {en_vocab_size:,}")
# print(f"  Vietnamese: {vi_vocab_size:,}")
# print(f"\nDataset info:")
# print(f"  Train samples: {len(all_train_sequences):,}")
# print(f"  Val samples: {len(all_val_sequences):,}")
# print(f"  Batches per epoch: {len(train_batches):,}")
# print(f"  Batch size: {REDUCED_BATCH_SIZE}")

# print("\n" + "="*60)
# print("INITIALIZING MODEL")
# print("="*60)

# # Initialize model
# model = Transformer(
#     src_pad_idx=PAD_TOKEN_POS,
#     trg_pad_idx=PAD_TOKEN_POS,
#     d_model=D_MODEL,
#     inp_vocab_size=vi_vocab_size,
#     trg_vocab_size=en_vocab_size,
#     max_len=MAX_SEQ_LEN,
#     d_ff=D_FF,
#     num_heads=NUM_HEADS,
#     num_layers=NUM_LAYERS,
#     dropout=DROPOUT,
#     device=DEVICE
# ).to(DEVICE)

# # ==================== LOAD CHECKPOINT IF RESUMING ====================
# if RESUME_TRAINING:
#     print("\n" + "="*60)
#     print("LOADING PRETRAINED MODEL")
#     print("="*60)
    
#     model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=DEVICE))
#     print(f"✓ Successfully loaded model from: {CHECKPOINT_PATH}")
    
#     # Extract best_loss from filename
#     import re
#     match = re.search(r'model-(\d+\.\d+)-', CHECKPOINT_PATH)
#     if match:
#         BEST_LOSS = float(match.group(1))
#         print(f"✓ Previous best validation loss: {BEST_LOSS:.3f}")
#     else:
#         BEST_LOSS = float('inf')
#         print("⚠ Could not extract loss from filename")
# else:
#     BEST_LOSS = float('inf')

# print(f'\nModel has {count_parameters(model):,} trainable parameters')
# print_gpu_memory()

# # ==================== OPTIMIZER ====================
# optimizer = optim.AdamW(
#     model.parameters(),
#     lr=0,
#     betas=(0.9, 0.98),
#     eps=1e-9,
#     weight_decay=1e-4
# )

# # ==================== SCHEDULER ====================
# if RESUME_TRAINING:
#     # Continue with normal training schedule
#     WARMUP_STEPS_USE = WARMUP_STEPS
#     MAX_LR_USE = 3e-4
    
#     scheduler = TransformerLRSchedule(
#         optimizer=optimizer,
#         d_model=D_MODEL,  
#         warmup_steps=WARMUP_STEPS_USE,
#         factor=FACTOR,
#         max_lr=MAX_LR_USE
#     )
    
#     # Skip the warmup steps already completed
#     batches_per_epoch = len(train_batches) // 2  # Account for accumulation
#     scheduler.current_step = TRAINED_EPOCHS * batches_per_epoch
#     print(f"\n✓ Scheduler resuming from step {scheduler.current_step:,}")
#     print(f"✓ Current learning rate: {scheduler.get_lr():.6f}")
# else:
#     scheduler = TransformerLRSchedule(
#         optimizer=optimizer,
#         d_model=D_MODEL,  
#         warmup_steps=WARMUP_STEPS,
#         factor=FACTOR,
#         max_lr=3e-4
#     )

# # ==================== LOSS FUNCTION ====================
# criterion = nn.CrossEntropyLoss(ignore_index=PAD_TOKEN_POS)

# # ==================== MIXED PRECISION SCALER ====================
# scaler = torch.amp.GradScaler('cuda')

# print("\n" + "="*60)
# print("TRAINING CONFIGURATION")
# print("="*60)
# print(f"  Actual batch size: {REDUCED_BATCH_SIZE}")
# print(f"  Gradient accumulation steps: 2")
# print(f"  Effective batch size: {REDUCED_BATCH_SIZE * 2}")
# print(f"  Weight updates per epoch: {len(train_batches) // 2}")
# print(f"  Total epochs: {EPOCHS}")

# print("\n" + "="*60)
# print("STARTING TRAINING")
# print("="*60)

# # Calculate remaining epochs
# if RESUME_TRAINING:
#     REMAINING_EPOCHS = EPOCHS - TRAINED_EPOCHS
#     print(f"\n⚠ Resuming from epoch {TRAINED_EPOCHS}")
#     print(f"⚠ Training {REMAINING_EPOCHS} additional epochs")
#     print(f"⚠ Total epochs will be: {EPOCHS}")
    
#     train_losses, test_losses = run(
#         total_epoch=REMAINING_EPOCHS,
#         best_loss=BEST_LOSS,
#         start_epoch=TRAINED_EPOCHS,
#         accumulation_steps=2
#     )
# else:
#     train_losses, test_losses = run(
#         total_epoch=EPOCHS, 
#         best_loss=BEST_LOSS,
#         start_epoch=0,
#         accumulation_steps=2
#     )

In [17]:
!pip install sacrebleu

In [18]:
# # ==================== STANDALONE BLEU EVALUATION SECTION ====================
# # Run this AFTER training is complete

# import torch
# from sacrebleu.metrics import BLEU

# print("\n" + "="*60)
# print("STANDALONE BLEU EVALUATION")
# print("="*60)

# # ==================== CONFIGURATION ====================
# EVAL_CHECKPOINT_PATH = '/kaggle/input/trained-models/model-2.612-0.471_40k_50k_8_epoches_vi_to_en.pt'  # Your trained model
# EVAL_BATCH_SIZE = 32  # Smaller batch for translation
# EVAL_MAX_LEN = 60  # Max translation length

# # ==================== VOCAB SIZE SETTINGS (MUST MATCH TRAINING!) ====================
# ENGLISH_VOCAB_SIZE = 40000   # MUST match what you used in training
# VIETNAMESE_VOCAB_SIZE = 50000  # MUST match what you used in training

# # ==================== TRANSLATION FUNCTIONS ====================

# def translate_sentence(model, src_tensor, vi_tokenizer, en_tokenizer, device, max_len=60):
#     """
#     Translate a single Vietnamese sentence to English using greedy decoding
#     """
#     model.eval()
    
#     with torch.no_grad():
#         # Add batch dimension if needed
#         if src_tensor.dim() == 1:
#             src_tensor = src_tensor.unsqueeze(0)
#         src_tensor = src_tensor.to(device)
        
#         # Start with START token
#         trg_indices = [en_tokenizer.word_index[START_TOKEN]]
        
#         for i in range(max_len):
#             trg_tensor = torch.LongTensor(trg_indices).unsqueeze(0).to(device)
            
#             with torch.amp.autocast('cuda'):
#                 output = model(src_tensor, trg_tensor)
            
#             # Get the predicted next token
#             pred_token = output.argmax(2)[:, -1].item()
#             trg_indices.append(pred_token)
            
#             # Stop if END token is predicted
#             if pred_token == en_tokenizer.word_index[END_TOKEN]:
#                 break
        
#         # Convert indices to words
#         trg_tokens = [en_tokenizer.index_word.get(idx, UNKNOWN_TOKEN) 
#                       for idx in trg_indices]
        
#         # Remove START and END tokens
#         trg_tokens = [token for token in trg_tokens 
#                       if token not in [START_TOKEN, END_TOKEN]]
        
#         return ' '.join(trg_tokens)

# def evaluate_bleu_corpus(model, data_loader, vi_tokenizer, en_tokenizer, device, 
#                          max_samples=None, verbose=True):
#     """
#     Calculate BLEU score on entire corpus
    
#     Args:
#         model: Trained translation model
#         data_loader: DataLoader with validation data
#         vi_tokenizer: Vietnamese tokenizer
#         en_tokenizer: English tokenizer
#         device: torch device
#         max_samples: Maximum number of samples to evaluate (None = all)
#         verbose: Print progress
    
#     Returns:
#         bleu_score: SacreBLEU score (0-100)
#         hypotheses: List of model translations
#         references: List of reference translations
#     """
#     model.eval()
    
#     hypotheses = []  # Model predictions
#     references = []  # Ground truth translations
    
#     total_samples = 0
    
#     if verbose:
#         print(f"\nTranslating validation set...")
#         print(f"Device: {device}")
    
#     with torch.no_grad():
#         for batch_idx, (src, trg) in enumerate(data_loader):
#             batch_size = src.size(0)
            
#             for j in range(batch_size):
#                 if max_samples and total_samples >= max_samples:
#                     break
                
#                 # Get source sentence (Vietnamese)
#                 src_sentence = src[j]
                
#                 # Translate Vietnamese -> English
#                 translation = translate_sentence(
#                     model, src_sentence, vi_tokenizer, en_tokenizer, device, max_len=EVAL_MAX_LEN
#                 )
#                 hypotheses.append(translation)
                
#                 # Get reference translation (English)
#                 trg_indices = trg[j].cpu().tolist()
#                 trg_tokens = []
#                 for idx in trg_indices:
#                     if idx == PAD_TOKEN_POS:
#                         continue
#                     if idx == en_tokenizer.word_index.get(START_TOKEN, -1):
#                         continue
#                     if idx == en_tokenizer.word_index.get(END_TOKEN, -1):
#                         break
#                     word = en_tokenizer.index_word.get(idx, '')
#                     if word:
#                         trg_tokens.append(word)
                
#                 reference = ' '.join(trg_tokens)
#                 references.append([reference])  # SacreBLEU expects list of lists
                
#                 total_samples += 1
            
#             if max_samples and total_samples >= max_samples:
#                 break
            
#             if verbose and (batch_idx + 1) % 20 == 0:
#                 print(f"  Processed {total_samples} sentences...")
                
#             # Memory cleanup
#             if (batch_idx + 1) % 50 == 0:
#                 torch.cuda.empty_cache()
    
#     if verbose:
#         print(f"  Total sentences translated: {total_samples}")
    
#     # Calculate BLEU score
#     bleu = BLEU()
#     bleu_score = bleu.corpus_score(hypotheses, references)
    
#     return bleu_score.score, hypotheses, references

# def print_translation_examples(hypotheses, references, num_examples=10):
#     """Print example translations"""
#     print(f"\n{'='*60}")
#     print("TRANSLATION EXAMPLES")
#     print(f"{'='*60}")
    
#     for i in range(min(num_examples, len(hypotheses))):
#         print(f"\nExample {i+1}:")
#         print(f"  Reference: {references[i][0]}")
#         print(f"  Predicted: {hypotheses[i]}")
#         print(f"  {'-'*58}")

# # ==================== RECREATE TOKENIZERS WITH VOCAB LIMITS ====================

# print("\n" + "="*60)
# print("RECREATING TOKENIZERS (40K EN / 50K VI)")
# print("="*60)

# print("⚠ Loading training data to recreate tokenizers...")
# print("⚠ This will take a few minutes...")

# en_data_train, vi_data_train = load_data(
#     "/kaggle/input/train-en-vi/train_2456580.en",
#     "/kaggle/input/train-en-vi/train_2456580.vi"
# )

# # Recreate tokenizers with EXPLICIT vocab limits
# def preprocess_tokenizer_with_limits(en_data, vi_data, en_vocab_size, vi_vocab_size):
#     """Create tokenizers with specified vocabulary size limits"""
#     print(f"Creating English tokenizer (limit: {en_vocab_size:,})...")
#     en_data, en_tokenizer = get_tokenize(
#         en_data, 
#         add_start_end=True,
#         max_vocab_size=en_vocab_size  # 40,000
#     )
    
#     print(f"Creating Vietnamese tokenizer (limit: {vi_vocab_size:,})...")
#     vi_data = [ViTokenizer.tokenize(sentence) for sentence in vi_data]
#     vi_data, vi_tokenizer = get_tokenize(
#         vi_data,
#         max_vocab_size=vi_vocab_size  # 50,000
#     )
    
#     return en_tokenizer, vi_tokenizer

# # Create tokenizers with vocab limits
# en_tokenizer, vi_tokenizer = preprocess_tokenizer_with_limits(
#     en_data_train, 
#     vi_data_train,
#     ENGLISH_VOCAB_SIZE,    # 40K
#     VIETNAMESE_VOCAB_SIZE  # 50K
# )

# en_vocab_size = len(en_tokenizer.word_index) + 1
# vi_vocab_size = len(vi_tokenizer.word_index) + 1

# print(f"✓ Tokenizers created successfully")
# print(f"✓ English vocab size: {en_vocab_size:,}")
# print(f"✓ Vietnamese vocab size: {vi_vocab_size:,}")

# # ==================== LOAD VALIDATION DATA ====================
# print("\n" + "="*60)
# print("LOADING VALIDATION DATA")
# print("="*60)

# en_data_val, vi_data_val = load_data(
#     data_path + "tst2013.en.txt", 
#     data_path + "tst2013.vi.txt"
# )

# en_val_sequences, vi_val_sequences = get_tokenize_seq(
#     en_data_val, vi_data_val, en_tokenizer, vi_tokenizer, 
#     max_sequence_length=MAX_SEQ_LEN
# )

# all_val_sequences = TranslationDataset(en_val_sequences, vi_val_sequences, MAX_SEQ_LEN)

# eval_loader = DataLoader(
#     all_val_sequences, 
#     batch_size=EVAL_BATCH_SIZE, 
#     shuffle=False,
#     pin_memory=False,
#     num_workers=0
# )

# print(f"✓ Validation samples: {len(all_val_sequences):,}")

# # ==================== LOAD TRAINED MODEL ====================

# print("\n" + "="*60)
# print("LOADING TRAINED MODEL FOR EVALUATION")
# print("="*60)

# # Initialize model with CORRECT vocab sizes from recreated tokenizers
# eval_model = Transformer(
#     src_pad_idx=PAD_TOKEN_POS,
#     trg_pad_idx=PAD_TOKEN_POS,
#     d_model=D_MODEL,
#     inp_vocab_size=vi_vocab_size,  # Uses recreated tokenizer (~50K)
#     trg_vocab_size=en_vocab_size,  # Uses recreated tokenizer (~40K)
#     max_len=MAX_SEQ_LEN,
#     d_ff=D_FF,
#     num_heads=NUM_HEADS,
#     num_layers=NUM_LAYERS,
#     dropout=DROPOUT,
#     device=DEVICE
# ).to(DEVICE)

# # Load checkpoint
# eval_model.load_state_dict(torch.load(EVAL_CHECKPOINT_PATH, map_location=DEVICE))
# eval_model.eval()

# print(f"✓ Model loaded from: {EVAL_CHECKPOINT_PATH}")
# print(f"✓ Model has {count_parameters(eval_model):,} parameters")

# # ==================== RUN EVALUATION ====================

# print("\n" + "="*60)
# print("CALCULATING BLEU SCORE")
# print("="*60)

# # Full evaluation on entire validation set
# bleu_score, hypotheses, references = evaluate_bleu_corpus(
#     model=eval_model,
#     data_loader=eval_loader,
#     vi_tokenizer=vi_tokenizer,
#     en_tokenizer=en_tokenizer,
#     device=DEVICE,
#     max_samples=None,  # Evaluate all samples (set to 500 for quick test)
#     verbose=True
# )

# # ==================== DISPLAY RESULTS ====================

# print("\n" + "="*60)
# print("EVALUATION RESULTS")
# print("="*60)
# print(f"\n✓ BLEU Score: {bleu_score:.2f}")
# print(f"✓ Total sentences evaluated: {len(hypotheses):,}")

# # Show translation examples
# print_translation_examples(hypotheses, references, num_examples=15)

# # ==================== ADDITIONAL METRICS (OPTIONAL) ====================

# print("\n" + "="*60)
# print("ADDITIONAL STATISTICS")
# print("="*60)

# # Calculate average translation length
# avg_hyp_len = sum(len(h.split()) for h in hypotheses) / len(hypotheses)
# avg_ref_len = sum(len(r[0].split()) for r in references) / len(references)

# print(f"Average hypothesis length: {avg_hyp_len:.1f} words")
# print(f"Average reference length: {avg_ref_len:.1f} words")
# print(f"Length ratio: {avg_hyp_len/avg_ref_len:.2f}")

# # Count empty translations
# empty_translations = sum(1 for h in hypotheses if len(h.strip()) == 0)
# print(f"Empty translations: {empty_translations} ({empty_translations/len(hypotheses)*100:.1f}%)")

# # Count <UNK> tokens (important for vocab-limited models)
# unk_count = sum(h.count(UNKNOWN_TOKEN) for h in hypotheses)
# print(f"Total <UNK> tokens in translations: {unk_count:,}")
# print(f"Average <UNK> per sentence: {unk_count/len(hypotheses):.2f}")

# print("\n" + "="*60)
# print("EVALUATION COMPLETE")
# print("="*60)

# # ==================== INTERACTIVE TRANSLATION (BONUS) ====================

# def translate_custom_sentence(sentence, model, vi_tokenizer, en_tokenizer, device):
#     """
#     Translate a custom Vietnamese sentence
#     """
#     # Tokenize Vietnamese
#     tokenized = ViTokenizer.tokenize(sentence)
#     sequence = vi_tokenizer.texts_to_sequences([tokenized])[0]
    
#     # Pad to max length
#     if len(sequence) < MAX_SEQ_LEN:
#         sequence = sequence + [PAD_TOKEN_POS] * (MAX_SEQ_LEN - len(sequence))
#     else:
#         sequence = sequence[:MAX_SEQ_LEN]
    
#     # Convert to tensor
#     src_tensor = torch.tensor(sequence, dtype=torch.long)
    
#     # Translate
#     translation = translate_sentence(model, src_tensor, vi_tokenizer, en_tokenizer, device)
    
#     return translation

# # Example: Translate custom sentences
# print("\n" + "="*60)
# print("CUSTOM TRANSLATION EXAMPLES")
# print("="*60)

# custom_sentences = [
#     "Tôi yêu học máy học.",
#     "Hôm nay thời tiết đẹp.",
#     "Chúng tôi đang làm việc trên dự án dịch máy."
# ]

# for sent in custom_sentences:
#     translation = translate_custom_sentence(sent, eval_model, vi_tokenizer, en_tokenizer, DEVICE)
#     print(f"\nVietnamese: {sent}")
#     print(f"English: {translation}")

# print("\n" + "="*60)

# Evaluate with beam search

In [19]:
!pip install sacrebleu

In [26]:
# ==================== STANDALONE MT EVALUATION (BEAM SEARCH) ====================
# Run this AFTER training is complete

import torch
import joblib
from sacrebleu.metrics import BLEU, CHRF
from torch.utils.data import DataLoader
from pyvi.ViTokenizer import ViTokenizer
from tqdm import tqdm

print("\n" + "=" * 60)
print("STANDALONE MT EVALUATION (BLEU + chrF++, BEAM SEARCH)")
print("=" * 60)

# ==================== CONFIGURATION ====================

EVAL_CHECKPOINT_PATH = "models/model-2.612-0.471_40k_50k_8_epoches_vi_to_en.pt"
TOKENIZER_PATH = "tokenizers/"

EVAL_BATCH_SIZE = 32
EVAL_MAX_LEN = 60
BEAM_SIZE = 3
LENGTH_PENALTY = 0.6

# ==================== LOAD TOKENIZERS ====================

print("\nLoading tokenizers used in training...")

en_tokenizer = joblib.load(TOKENIZER_PATH + "vi_en_en_tokenizer_trg_40k.pkl")
vi_tokenizer = joblib.load(TOKENIZER_PATH + "vi_en_vi_tokenizer_src_50k.pkl")

en_vocab_size = len(en_tokenizer.word_index) + 1
vi_vocab_size = len(vi_tokenizer.word_index) + 1

print(f"✓ English vocab size: {en_vocab_size:,}")
print(f"✓ Vietnamese vocab size: {vi_vocab_size:,}")

START_ID = en_tokenizer.word_index[START_TOKEN]
END_ID = en_tokenizer.word_index[END_TOKEN]

# ==================== BEAM SEARCH DECODING ====================

def beam_search_translate(model, src_tensor, device, max_len=60, beam_size=5):
    model.eval()

    if src_tensor.dim() == 1:
        src_tensor = src_tensor.unsqueeze(0)
    src_tensor = src_tensor.to(device)

    # beam = (token_ids, log_prob)
    beams = [([START_ID], 0.0)]
    completed = []

    with torch.no_grad():
        for _ in range(max_len):
            new_beams = []

            for tokens, score in beams:
                if tokens[-1] == END_ID:
                    completed.append((tokens, score))
                    continue

                trg_tensor = torch.LongTensor(tokens).unsqueeze(0).to(device)
                output = model(src_tensor, trg_tensor)

                log_probs = torch.log_softmax(output[0, -1], dim=-1)
                topk = torch.topk(log_probs, beam_size)

                for i in range(beam_size):
                    new_tokens = tokens + [topk.indices[i].item()]
                    new_score = score + topk.values[i].item()
                    new_beams.append((new_tokens, new_score))

            beams = sorted(
                new_beams,
                key=lambda x: x[1] / (len(x[0]) ** LENGTH_PENALTY),
                reverse=True
            )[:beam_size]

        completed.extend(beams)

    best_tokens = max(
        completed,
        key=lambda x: x[1] / (len(x[0]) ** LENGTH_PENALTY)
    )[0]

    words = [
        en_tokenizer.index_word.get(t, UNKNOWN_TOKEN)
        for t in best_tokens
        if t not in (START_ID, END_ID)
    ]

    return " ".join(words)

# ==================== EVALUATION ====================
def evaluate_corpus(model, data_loader, device):
    hypotheses = []
    references = []

    model.eval()

    total_sentences = len(data_loader.dataset)
    pbar = tqdm(
        total=total_sentences,
        desc="🔄 Translating (beam search)",
        unit="sent"
    )

    with torch.no_grad():
        for src, trg in data_loader:
            batch_size = src.size(0)

            for j in range(batch_size):
                hyp = beam_search_translate(
                    model, src[j], device, EVAL_MAX_LEN, BEAM_SIZE
                )
                hypotheses.append(hyp)

                ref_tokens = []
                for idx in trg[j].cpu().tolist():
                    if idx in (PAD_TOKEN_POS, START_ID):
                        continue
                    if idx == END_ID:
                        break
                    w = en_tokenizer.index_word.get(idx, "")
                    if w:
                        ref_tokens.append(w)

                references.append([" ".join(ref_tokens)])

                pbar.update(1)

    pbar.close()

    bleu = BLEU()
    chrf = CHRF(word_order=2)

    bleu_score = bleu.corpus_score(hypotheses, references)
    chrf_score = chrf.corpus_score(hypotheses, references)

    return bleu_score, chrf_score, hypotheses, references


def print_examples(hypotheses, references, n=10):
    print("\n" + "=" * 60)
    print("TRANSLATION EXAMPLES")
    print("=" * 60)

    for i in range(min(n, len(hypotheses))):
        print(f"\nExample {i+1}:")
        print("REF:", references[i][0])
        print("HYP:", hypotheses[i])

# ==================== LOAD VALIDATION DATA ====================

print("\nLoading validation data...")

en_data_val, vi_data_val = load_data(
    data_path + "tst2013.en.txt",
    data_path + "tst2013.vi.txt"
)

en_val_seq, vi_val_seq = get_tokenize_seq(
    en_data_val,
    vi_data_val,
    en_tokenizer,
    vi_tokenizer,
    max_sequence_length=MAX_SEQ_LEN
)

val_dataset = TranslationDataset(en_val_seq, vi_val_seq, MAX_SEQ_LEN)

val_loader = DataLoader(
    val_dataset,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

print(f"✓ Validation samples: {len(val_dataset):,}")

# ==================== LOAD MODEL ====================

print("\nLoading trained model...")

model = Transformer(
    src_pad_idx=PAD_TOKEN_POS,
    trg_pad_idx=PAD_TOKEN_POS,
    d_model=D_MODEL,
    inp_vocab_size=vi_vocab_size,
    trg_vocab_size=en_vocab_size,
    max_len=MAX_SEQ_LEN,
    d_ff=D_FF,
    num_heads=NUM_HEADS,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
    device=DEVICE
).to(DEVICE)

model.load_state_dict(torch.load(EVAL_CHECKPOINT_PATH, map_location=DEVICE))
model.eval()

print("✓ Model loaded")

# ==================== RUN EVALUATION ====================

bleu, chrf, hyps, refs = evaluate_corpus(
    model,
    val_loader,
    DEVICE
)

print("\n" + "=" * 60)
print(f"BLEU  : {bleu.score:.2f}")
print(f"chrF++: {chrf.score:.2f}")
print("=" * 60)

print_examples(hyps, refs, n=10)



STANDALONE MT EVALUATION (BLEU + chrF++, BEAM SEARCH)

Loading tokenizers used in training...
✓ English vocab size: 40,004
✓ Vietnamese vocab size: 50,002

Loading validation data...
✓ Validation samples: 1,228

Loading trained model...
✓ Model loaded


🔄 Translating (beam search):   0%|          | 1/1228 [00:07<2:41:23,  7.89s/sent]

KeyboardInterrupt: 